# Homework 2: Vector Search

LLM Zoomcamp – Module 2 Homework


## Setup

Dieses Notebook nutzt den ONNX `Embedder` aus `embedder.py`.  
Voraussetzung: `download.py` wurde bereits ausgeführt und der Ordner `models/` existiert.

In [1]:
import sys
print(sys.executable)

/Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/.venv/bin/python


In [2]:
import os

os.chdir("/Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/llm-zoomcamp-hw2")

print(os.getcwd())
print(os.listdir())

/Users/veneraheddergott/LLM_zoomcamp/llm-zoomcamp_2026_vh-1/llm-zoomcamp-hw2
['homework_hw2.ipynb', 'uv.lock', 'faq_vectors2.db-wal', 'download.py', 'pyproject.toml', 'models', '__pycache__', 'README.md', '.venv', '.python-version', 'embedder.py', 'faq_vectors2.db', '.ipynb_checkpoints', 'main.py']


In [3]:
import sys
!{sys.executable} -m pip install "onnxruntime==1.17.3" tokenizers numpy tqdm minsearch gitsource


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [4]:
from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
import numpy as np
from tqdm.auto import tqdm

embed = Embedder()
print('Embedder loaded')

Embedder loaded


## Q1. Embedding a query

Query:

> How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (v[0])?


In [5]:
query_q1 = 'How does approximate nearest neighbor search work?'
v = embed.encode(query_q1)

print('Shape:', v.shape)
print('v[0]:', v[0])

Shape: (384,)
v[0]: -0.02058200593003704


### Loading the Data
We pull the lesson pages from the course repository, the same way as in homework 1. We pin to commit 8c1834d so everyone works with the same data.

In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [7]:
len(documents)

72

## Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?


In [8]:
target_filename = '02-vector-search/lessons/07-sqlitesearch-vector.md'

doc = [d for d in documents if d['filename'] == target_filename][0]
doc_vector = embed.encode(doc['content'])
similarity = doc_vector.dot(v)

print('Similarity:', similarity)

Similarity: 0.3610702814461231


## Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [10]:
texts = [chunk["content"] for chunk in chunks]

X = embed.encode_batch(texts)

In [11]:
scores = X.dot(v)

In [12]:
idx = scores.argmax()
chunks[idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

## Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

What metric do we use to evaluate a search engine?
Which file is the filename of the first result?

02-vector-search/lessons/04-vector-search.md
04-evaluation/lessons/05-search-metrics.md
04-evaluation/lessons/13-llm-as-judge.md
05-monitoring/lessons/04-metrics.md

In [13]:
from minsearch import VectorSearch

index = VectorSearch(keyword_fields=["filename"])
index.fit(X, chunks)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [14]:
query = "What metric do we use to evaluate a search engine?"
v_query = embed.encode(query)

In [15]:
results = index.search(v_query, num_results=5)

In [16]:
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

In [17]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

## Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

Run both searches for this query:

How do I store vectors in PostgreSQL?
Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

02-vector-search/lessons/01-intro.md
02-vector-search/lessons/02-embeddings.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md

In [18]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

In [19]:
query = "How do I store vectors in PostgreSQL?"
text_results = text_index.search(query, num_results=5)

In [20]:
v_query = embed.encode(query)

vector_results = vindex.search(v_query, num_results=5)

In [21]:
text_files = [r["filename"] for r in text_results]
vector_files = [r["filename"] for r in vector_results]

print("Text search:")
for f in text_files:
    print(f)

print()
print("Vector search:")
for f in vector_files:
    print(f)

Text search:
02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md

Vector search:
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [22]:
for f in vector_files:
    if f not in text_files:
        print(f)

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


### Q6. Hybrid search

In [23]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [24]:
results = rrf([vector_results, text_results])

In [25]:
query = "How do I give the model access to tools?"

In [26]:
#vector search
v_query = embed.encode(query)

vector_results = vindex.search(v_query, num_results=5)

In [27]:
#text search
text_results = text_index.search(query, num_results=5)

In [30]:
print("Vector results:")
for r in vector_results:
    print(r["filename"])

print()
print("Text results:")
for r in text_results:
    print(r["filename"])

Vector results:
01-agentic-rag/lessons/01-intro.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/16-other-frameworks.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md

Text results:
01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
04-evaluation/lessons/02-ground-truth.md


In [31]:
results = rrf([vector_results, text_results])

In [32]:
results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'